In [56]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('datefruit_ds.csv')
df.isnull().sum()
df.info()
df.sample(2)
df['Class'].unique()

In [101]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

le = LabelEncoder()
df['Class'] = le.fit_transform(df['Class'])

X = df.drop(['Class'], axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [102]:
# from sklearn.decomposition import PCA

# pca = PCA(n_components=30)
# X_train_pca = pca.fit_transform(X_train_scaled)
# X_test_pca = pca.transform(X_test_scaled)

In [103]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
X_train.shape
# X_test.shape

(718, 30)

In [105]:

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model=nn.Sequential(

            nn.Linear(X_train.shape[1], 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, 7),
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [106]:
import torch.optim as optim

model = ANN()

criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters())

In [107]:

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimiser.zero_grad()

        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f'Epoch {epoch+1} / {epochs} train loss : {train_loss}')

Epoch 1 / 100 train loss : 1.7408718337183413
Epoch 2 / 100 train loss : 1.148809295633565
Epoch 3 / 100 train loss : 0.7677938458712205
Epoch 4 / 100 train loss : 0.5560695451238881
Epoch 5 / 100 train loss : 0.4525046795606613
Epoch 6 / 100 train loss : 0.39307836216429004
Epoch 7 / 100 train loss : 0.34311952020811
Epoch 8 / 100 train loss : 0.30990883902363153
Epoch 9 / 100 train loss : 0.28702357346596924
Epoch 10 / 100 train loss : 0.26572339755037555
Epoch 11 / 100 train loss : 0.24714736737634824
Epoch 12 / 100 train loss : 0.2358047631771668
Epoch 13 / 100 train loss : 0.214449317558952
Epoch 14 / 100 train loss : 0.20595580857733023
Epoch 15 / 100 train loss : 0.1992883474930473
Epoch 16 / 100 train loss : 0.19360480749088785
Epoch 17 / 100 train loss : 0.17769996012034622
Epoch 18 / 100 train loss : 0.17277141320316688
Epoch 19 / 100 train loss : 0.167097398120424
Epoch 20 / 100 train loss : 0.1596303714196319
Epoch 21 / 100 train loss : 0.15320072938566623
Epoch 22 / 100 tr

In [108]:
correct = 0
total = 0


model.eval()

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        # max_val, index = torch.max(outputs, 1)
        _, prediction = torch.max(outputs, 1)

        correct += (prediction == yb).sum().item()
        total += yb.size(0)

print(f'Correct vals: {correct}')
print(f'Total vals: {total}')
print(f'Accuracy: {correct / total * 100}')

Correct vals: 171
Total vals: 180
Accuracy: 95.0
